# Practice Lab: RAG with LlamaIndex (Lesson 8.6)

Module 8 · 8 exercises · VectorStoreIndex + synthesis modes + IngestionPipeline + SubQuestion

Exercises 1, 2, 3, 5 run locally (pure Python simulations).


# Lesson 8.8: RAG with LlamaIndex — The Data Framework

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 3, 5 run **locally** (pure Python simulations of LlamaIndex patterns).


In [ ]:
import numpy as np
import hashlib, math
from collections import Counter


---
## Exercise 1 (Easy): LlamaIndex RAG in 5 Lines


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib
import numpy as np

def fake_embed(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    v = np.array([b/255.0 for b in h[:dim]])
    for w, i in {"refund":0,"policy":1,"price":2,"course":4,"prereq":5,"python":6,"live":8,"youtube":10}.items():
        if w in t.lower() and i < dim: v[i] += 0.5
    return v / np.linalg.norm(v)

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

class Document:
    def __init__(self, text, metadata=None):
        self.text, self.metadata = text, metadata or {}

class VectorStoreIndex:
    def __init__(self): self.nodes, self.embs = [], []
    @classmethod
    def from_documents(cls, docs):
        idx = cls()
        for d in docs:
            idx.nodes.append({"text": d.text, "meta": d.metadata})
            idx.embs.append(fake_embed(d.text))
        print(f"  Indexed {len(idx.nodes)} nodes (auto: chunk+embed+store)")
        return idx
    def as_query_engine(self, top_k=2):
        return QueryEngine(self, top_k)
    def as_chat_engine(self, mode="condense_plus_context"):
        return ChatEngine(self, mode)

class QueryEngine:
    def __init__(self, idx, k): self.idx, self.k = idx, k
    def query(self, q):
        qe = fake_embed(q)
        scores = sorted([(i, cosine(qe, e)) for i, e in enumerate(self.idx.embs)], key=lambda x: -x[1])[:self.k]
        top = self.idx.nodes[scores[0][0]]
        return {"answer": f"Based on context: {top['text'][:60]}...", "sources": [self.idx.nodes[i]["meta"] for i, _ in scores]}

class ChatEngine:
    def __init__(self, idx, mode): self.engine = QueryEngine(idx, 2); self.history = []; self.mode = mode
    def chat(self, msg):
        resolved = msg
        if self.history and any(p in msg.lower() for p in ["it","that","its"]):
            resolved = f"{self.history[-1]} {msg}"
        self.history.append(msg)
        return self.engine.query(resolved)

docs = [Document("Refund policy: full refund 7 days. 50% 8-30 days.", {"source":"refund.txt"}),
        Document("GenAI course: 14999 rupees lifetime access certificate.", {"source":"pricing.txt"}),
        Document("Live classes daily 7 PM IST YouTube recordings.", {"source":"schedule.txt"}),
        Document("Prerequisites: basic Python high school math.", {"source":"prereqs.txt"})]

print("LlamaIndex RAG (simulated):")
index = VectorStoreIndex.from_documents(docs)
engine = index.as_query_engine(top_k=2)
for q in ["Refund policy?", "Prerequisites?", "Live class time?"]:
    r = engine.query(q)
    print(f"  Q: {q} -> {r['answer'][:50]}... Sources: {r['sources'][:1]}")

print(f"\nChat Engine:")
chat = index.as_chat_engine()
for msg in ["What is the course?", "How much does it cost?"]:
    r = chat.chat(msg)
    print(f"  {msg} -> {r['answer'][:45]}...")

print(f"\nLines: Scratch ~60, LangChain ~15, LlamaIndex ~5")


</details>


---
## Exercise 2 (Easy): Response Synthesis Comparison


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import math

chunks = ["Netsetos edtech Hyderabad", "14999 rupees lifetime", "Refund 7 days 50% 8-30", "7 PM IST YouTube", "Python math prereqs"]

def sim_mode(name, chunks):
    n = len(chunks)
    modes = {"compact": (min(2, n//3+1), "Stuff all, 1-2 calls"),
             "refine": (n, "Sequential, N calls"),
             "tree_summarize": (max(1, math.ceil(math.log2(n))+1), "Tree merge, log(N)"),
             "simple_summarize": (1, "Truncate, 1 call (DATA LOSS)"),
             "accumulate": (n, "Per-chunk, no synthesis"),
             "compact_accumulate": (max(1,n//2), "Grouped, no synthesis")}
    calls, desc = modes[name]
    return {"mode": name, "calls": calls, "desc": desc}

print("Response Synthesis Modes (5 chunks):")
print(f"{'Mode':<22} {'Calls':<8} {'Description'}")
print("=" * 55)
for m in ["compact","refine","tree_summarize","simple_summarize","accumulate","compact_accumulate"]:
    r = sim_mode(m, chunks)
    print(f"  {r['mode']:<20} {r['calls']:<8} {r['desc']}")

print(f"\nRecommendations:")
for use, mode in [("Factoid QA","compact"),("Summarization","tree_summarize"),("Legal/compliance","refine"),("Per-source","accumulate")]:
    print(f"  {use:<18} {mode}")
print(f"Streaming: only last LLM call streams in multi-call modes")


</details>


---
## Exercise 3 (Medium): IngestionPipeline with Caching


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib

class IngestionPipeline:
    def __init__(self, transforms):
        self.transforms = transforms; self.cache = {}
    def _hash(self, text): return hashlib.md5(text.encode()).hexdigest()[:12]
    def run(self, docs, doc_store=None):
        stats = {"processed": 0, "cached": 0, "total": 0}
        for doc in docs:
            stats["total"] += 1
            dh = self._hash(doc["text"])
            if doc_store is not None:
                if doc["id"] in doc_store and doc_store[doc["id"]] == dh:
                    stats["cached"] += 1; continue
                doc_store[doc["id"]] = dh
            node = doc["text"]; all_cached = True
            for t_name, t_fn in self.transforms:
                ck = (dh, t_name)
                if ck in self.cache: node = self.cache[ck]
                else: node = t_fn(node); self.cache[ck] = node; all_cached = False
            if all_cached: stats["cached"] += 1
            else: stats["processed"] += 1
        return stats

pipe = IngestionPipeline([("split", lambda t: t[:100]), ("embed", lambda t: f"[emb]{t[:50]}")])
ds = {}
v1 = [{"id":f"d{i}","text":t} for i, t in enumerate(["Refund 7 days","14999 rupees","7PM IST","Python math","EMI 2500"])]
print(f"Run 1 (initial): {pipe.run(v1, ds)}")
print(f"Run 2 (same):    {pipe.run(v1, ds)}")
v2 = [{"id":"d0","text":"Refund 14 days"},{"id":"d1","text":"19999 rupees"},{"id":"d2","text":"7PM IST"},{"id":"d3","text":"Python math"},{"id":"d4","text":"EMI 2500"}]
print(f"Run 3 (2 mod):   {pipe.run(v2, ds)}")
print(f"\nDedup: UPSERTS (ID), DUPLICATES_ONLY (hash), UPSERTS_AND_DELETE (sync)")
print(f"Production: RedisKVStore cache + RedisDocumentStore dedup")


</details>


---
## Exercise 5 (Medium): SubQuestionQueryEngine


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib
import numpy as np

def fe(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    v = np.array([b/255.0 for b in h[:dim]])
    for w, i in {"netsetos":0,"campusx":1,"price":2,"refund":3,"14999":6,"9999":7}.items():
        if w in t.lower() and i < dim: v[i] += 0.5
    return v / np.linalg.norm(v)

kb_a = ["Netsetos GenAI: 14999 rupees 14 modules 146 hours", "Netsetos refund: full 7 days 50% 8-30", "Netsetos: Discord live sessions GCP credits"]
kb_b = ["CampusX ML: 9999 rupees 10 modules 80 hours", "CampusX refund: full 3 days none after", "CampusX: Telegram group recorded sessions"]

def search(q, kb):
    qe = fe(q)
    return max(kb, key=lambda d: np.dot(qe, fe(d))/(np.linalg.norm(qe)*np.linalg.norm(fe(d))))

class SubQEngine:
    def __init__(self, tools): self.tools = tools
    def query(self, q):
        subs = []
        for name, kb in self.tools.items():
            subs.append({"tool": name, "answer": search(q, kb)})
        return {"subs": subs, "synthesis": " vs ".join(s["answer"][:40] for s in subs)}

engine = SubQEngine({"Netsetos": kb_a, "CampusX": kb_b})
for q in ["Compare pricing", "Compare refund policies", "Which has more hours?"]:
    r = engine.query(q)
    print(f"  Q: {q}")
    for s in r["subs"]: print(f"    [{s['tool']}] {s['answer'][:45]}...")
    print(f"    Synthesis: {r['synthesis'][:55]}...")

print(f"\nKey: specific tool descriptions drive routing quality")
print(f"use_async=True for parallel sub-question execution")


</details>


---
## Exercise 4 (Medium): RouterQueryEngine
Architecture/design. See practice-lab-8_8.html.


In [ ]:
# Exercise 4: RouterQueryEngine
pass


---
## Exercise 6 (Challenge): FunctionAgent with Tools
Architecture/design. See practice-lab-8_8.html.


In [ ]:
# Exercise 6: FunctionAgent with Tools
pass


---
## Exercise 7 (Challenge): LlamaParse + Evaluation Pipeline
Architecture/design. See practice-lab-8_8.html.


In [ ]:
# Exercise 7: LlamaParse + Evaluation Pipeline
pass


---
## Exercise 8 (Challenge): LlamaIndex + LangChain Hybrid
Architecture/design. See practice-lab-8_8.html.


In [ ]:
# Exercise 8: LlamaIndex + LangChain Hybrid
pass
